# 🔍 Unfallatlas Deutschland — U-Phase: Daten verstehen

Bevor ein Modell trainiert wird, müssen die Daten sprechen. In der **U-Phase** — *Understanding the Data* — untersuchen wir, was der Unfallatlas tatsächlich enthält: Schema, Verteilungen, räumliche Muster, Zeitstrukturen und Datenlücken.

Erst wenn wir verstehen, *was* in den Daten steckt, können wir sinnvolle Features bauen und fundierte Modellentscheidungen treffen.

---

## Die Position im QUA³CK-Prozess

| Phase | Notebook | Inhalt | Status |
| :--- | :--- | :--- | :---: |
| **Q** — Question | `01_Q_Phase.ipynb` | Forschungsfrage, Hypothesen, Metriken, Literatur | ✅ |
| **→ U** — Understanding | `02_U_Phase.ipynb` | EDA, Geo-Visualisierung, Feature Engineering, DWD-Join | 🔄 |
| **A³** — Algorithm / Adapt / Adjust | `03_A3_Phase.ipynb` | Baselines, Boosting-Modelle, Imbalance-Strategien, Tuning | 🔄 |
| **C** — Conclude & Compare | `04_C_Phase.ipynb` | SHAP, Modellvergleich, Limitationen, Fazit | 🔄 |
| **K** — Knowledge Transfer | `app/streamlit_app.py` | Interaktive Risikoprofil-App (Streamlit) | 🔄 |

---

In [ ]:
# ============================================================
# Bibliotheken importieren
# Alle Pakete werden einmal am Anfang geladen — das ist
# gute Praxis und vermeidet versteckte Abhängigkeiten.
# ============================================================

from pathlib import Path    # Plattformunabhängige Dateipfade (Windows & Linux)
import duckdb               # SQL-Engine für Parquet-Dateien — schneller als pandas für 2 Mio. Zeilen
import pandas as pd         # Tabellengliche Daten verarbeiten und anzeigen

# ============================================================
# Pfade konfigurieren
# Wir arbeiten relativ zum Notebook-Verzeichnis, damit das
# Projekt auf jedem Rechner ohne Anpassung funktioniert.
# ============================================================
BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA     = BASE_DIR / "data" / "body.parquet"

# Plausibilitätsprüfung: Existiert die Datendatei?
if not DATA.exists():
    raise FileNotFoundError(
        f"Datendatei nicht gefunden: {DATA.resolve()}\n"
        "Bitte sicherstellen, dass body.parquet im data/-Verzeichnis liegt."
    )

# DuckDB-Verbindung (in-memory, Parquet wird direkt abgefragt)
con = duckdb.connect()

print(f"✓ DuckDB  {duckdb.__version__}")
print(f"✓ Pandas  {pd.__version__}")
print(f"✓ Datei   {DATA.resolve()}")
print(f"✓ Größe   {DATA.stat().st_size / 1_048_576:.1f} MB")

Die Umgebung ist bereit. Die Datei `body.parquet` (~66 MB) enthält den **konsolidierten Unfallatlas** — alle verfügbaren Jahrgänge in einem kompakten Parquet-Format, direkt abfragbar via DuckDB ohne vorheriges Einlesen in den Arbeitsspeicher.

---

## 0 — Datensatz-Überblick

Wir starten mit einer Schema-Inspektion: Welche Spalten gibt es, welche Datentypen haben sie, und welche zeitliche Abdeckung hat der Datensatz?

In [ ]:
# ============================================================
# Spalten und Datentypen inspizieren
# DESCRIBE gibt einen vollständigen Überblick über das Schema
# — vergleichbar mit df.info() in pandas, aber für Parquet.
# ============================================================
schema = con.execute(f"DESCRIBE SELECT * FROM '{DATA}'").df()
print(f"Spaltenanzahl: {len(schema)}")
print()
schema

**Was das Schema zeigt:**

- **21 Spalten** — Zeit (`UJAHR`, `UMONAT`, `USTUNDE`, `UWOCHENTAG`), Schwere (`UKATGEORIE`), Unfallart (`UART`, `UTYP1`), Umfeld (`ULICHTVERH`, `STRZUSTAND`), Verkehrsmittel (6 binäre Flags) und Geo-Koordinaten (`LON`, `LAT`).
- Kein `ULAND`-Feld — das Bundesland wird aus den ersten zwei Ziffern von `UKREIS` abgeleitet.
- Koordinaten als `LON`/`LAT` (WGS84, Dezimalgrad) — abweichend von der offiziellen Dokumentation (`XGCSWGS84`/`YGCSWGS84`).
- **Tippfehler:** Die Zielvariable heißt `UKATGEORIE` (fehlendes zweites K) — alle Skripte verwenden diesen tatsächlichen Namen.

### 📊 Spaltenschema (verifiziert)

| Spalte | Typ | Bedeutung | Kodierung |
| :--- | :---: | :--- | :--- |
| `OBJECTID` | INTEGER | Eindeutige Unfall-ID | — |
| `UJAHR` | SMALLINT | Unfalljahr | 2016–2024 |
| `UMONAT` | TINYINT | Unfallmonat | 1–12 |
| `USTUNDE` | TINYINT | Unfallstunde | 0–23 |
| `UWOCHENTAG` | TINYINT | Wochentag | 1 = Sonntag · 2 = Montag · … · 7 = Samstag |
| **`UKATGEORIE`** | **TINYINT** | **Zielvariable: Unfallschwere** | **1 = Getötet · 2 = Schwer verletzt · 3 = Leicht verletzt** |
| `UART` | TINYINT | Unfallart | 0–9 (10 Klassen, z. B. Auffahrunfall, Abkommen) |
| `UTYP1` | TINYINT | Unfalltyp | 1–7 (z. B. Fahrradunfall, Fußgängerunfall) |
| `ULICHTVERH` | TINYINT | Lichtverhältnisse | 0 = Tageslicht · 1 = Dämmerung · 2 = Dunkelheit |
| `STRZUSTAND` | TINYINT | Straßenzustand | 0 = trocken · 1 = nass/feucht · 2 = winterglatt |
| `IstRad` | BOOLEAN | Fahrradbeteiligung | True / False |
| `IstPKW` | BOOLEAN | PKW-Beteiligung | True / False |
| `IstFuss` | BOOLEAN | Fußgängerbeteiligung | True / False |
| `IstKrad` | BOOLEAN | Krad / Motorrad | True / False |
| `IstGkfz` | BOOLEAN | Güterkraftfahrzeug | True / False |
| `IstSonstig` | BOOLEAN | Sonstiges Verkehrsmittel | True / False |
| `LON` | DOUBLE | Längengrad WGS84 | Dezimalgrad (5,87–15,03) |
| `LAT` | DOUBLE | Breitengrad WGS84 | Dezimalgrad (47,31–55,05) |
| `UREGBEZ` | VARCHAR | Regierungsbezirk-Code | — |
| `UKREIS` | VARCHAR | Kreis-Code (5-stellig) | Erste 2 Ziffern = Bundesland-Schlüssel |
| `UGEMEINDE` | VARCHAR | Gemeinde-Code | — |

In [ ]:
# ============================================================
# Gesamtgröße und zeitliche Abdeckung
# ============================================================
gesamt = con.execute(
    f"SELECT COUNT(*) AS zeilen_gesamt, COUNT(DISTINCT UJAHR) AS jahrgaenge,"
    f" MIN(UJAHR) AS erstes_jahr, MAX(UJAHR) AS letztes_jahr FROM '{DATA}'"
).df()

print("── Gesamtübersicht ────────────────────────────────")
print(gesamt.to_string(index=False))

print()
print("── Unfälle pro Jahr ───────────────────────────────")
jahre = con.execute(
    f"SELECT UJAHR AS jahr, COUNT(*) AS unfaelle,"
    f" ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS anteil_pct"
    f" FROM '{DATA}' GROUP BY UJAHR ORDER BY UJAHR"
).df()
print(jahre.to_string(index=False))

**Was die Jahresverteilung zeigt:**

Alle **neun Jahrgänge 2016–2024** sind lückenlos abgedeckt — 2,09 Millionen Unfälle insgesamt. Der **Rückgang 2020** ist auf die COVID-19-Pandemie zurückzuführen: deutlich weniger Pendler- und Freizeitverkehr. Ab 2021 erholt sich das Niveau und stabilisiert sich auf ~268.000–269.000 Unfälle pro Jahr — konsistent mit den BASt-Jahresberichten und damit ein erster Vertrauensindikator für die Datenqualität.

---

## 1 — Zielvariable: Unfallschwere

Die Zielvariable `UKATGEORIE` ist das Label, das das Modell später vorhersagen soll. Hier untersuchen wir ihre Verteilung und quantifizieren das Imbalance-Problem.

In [ ]:
# ============================================================
# Zielvariable: UKATGEORIE — Unfallschwere
# Das ist das Label, das das Modell später vorhersagen soll.
# ============================================================
ziel = con.execute(
    f"SELECT UKATGEORIE AS klasse,"
    f" CASE UKATGEORIE WHEN 1 THEN '1 — Getötet'"
    f" WHEN 2 THEN '2 — Schwer verletzt'"
    f" ELSE '3 — Leicht verletzt' END AS schweregrad,"
    f" COUNT(*) AS n,"
    f" ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS anteil_pct"
    f" FROM '{DATA}' GROUP BY UKATGEORIE ORDER BY UKATGEORIE"
).df()

print("── Zielvariable UKATGEORIE ───────────────────────")
print(ziel.to_string(index=False))

**Kernbefund — Starke Klassenimbalance:**

| Klasse | Schweregrad | Anteil |
| :---: | :--- | :---: |
| 1 | Getötet | ~1 % |
| 2 | Schwer verletzt | ~18 % |
| 3 | Leicht verletzt | ~81 % |

> **⚠️ Einfach erklärt:** Stellen Sie sich vor, Sie trainieren ein Modell mit 100 Beispielen — 1 davon zeigt einen tödlichen Unfall, 81 zeigen leichte. Ein Modell, das *immer* „leicht" antwortet, hat 81 % Trefferquote — und ist trotzdem wertlos, weil es den einen Todesfall komplett übersieht. Genau deshalb verwenden wir **macro-F1** als Metrik, die alle Klassen gleich gewichtet, und **Imbalance-Strategien** wie Class Weights und SMOTE in der A³-Phase.

---

## 2 — Train / Val / Test Split

Der chronologische Split wird hier empirisch verifiziert — wir prüfen, ob die Jahrgänge die erwarteten Größenverhältnisse liefern.

In [ ]:
# ============================================================
# Train / Val / Test Split — Größen verifizieren
# ============================================================
split = con.execute(
    f"SELECT "
    f" CASE WHEN UJAHR <= 2022 THEN 'Train  (2016–2022)'"
    f" WHEN UJAHR = 2023 THEN 'Val    (2023)'"
    f" ELSE 'Test   (2024)' END AS split,"
    f" MIN(UJAHR) AS von, MAX(UJAHR) AS bis,"
    f" COUNT(*) AS n,"
    f" ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS anteil_pct"
    f" FROM '{DATA}' GROUP BY split ORDER BY MIN(UJAHR)"
).df()

print("── Chronologischer Train / Val / Test Split ─────")
print(split.to_string(index=False))

Das Testset 2024 umfasst ~268.000 Unfälle — groß genug für statistisch belastbare Aussagen, aber vollständig von der Modellentwicklung getrennt. Es wird **einmalig** zur finalen Bewertung in der C-Phase verwendet.

---

## 3 — Explorative Datenanalyse (EDA)

> 🔄 **TODO** — Folgende Analysen sind geplant:
> - Verteilungsplots für alle kategorialen Features (ULICHTVERH, STRZUSTAND, UART, UTYP1)
> - Heatmap Wochentag × Stunde × mittlere Unfallschwere (H3)
> - Stundenprofil: Häufigkeit + mittlere Schwere (H7)
> - Cramér's V Korrelationsmatrix aller kategorialen Features vs. UKATGEORIE (H1, H2)
> - Verkehrsmittel-Beteiligung nach Schweregrad (H4)

## 4 — Geo-Visualisierung

> 🔄 **TODO** — Folgende Visualisierungen sind geplant:
> - Folium-Heatmap: Alle 2,09 Mio. Unfälle nach Dichte (Subsampling 10 %)
> - Choroplethenkarte: Mittlere Unfallschwere pro Bundesland → Überprüfung H5
> - Zoom: Hessen / Wiesbaden als regionale Fallstudie
> - ULAND-Ableitung: `df["ULAND"] = df["UKREIS"].str[:2].astype(int)`

## 5 — DWD-Wetter-Join

> 🔄 **TODO** — Geplante Wetterdaten-Anreicherung via `wetterdienst`:
> - **Parameter:** air_temperature · visibility · weather_phenomena
> - **Stationen:** ~130 Synop-Stationen mit allen drei Parametern
> - **Zeitraum:** 2016-01-01 → 2024-12-31, stündliche Auflösung
> - **Join-Strategie:** Nächste Station via scipy cKDTree + Zeitjoin via DuckDB
> - **Output:** `data/interim/accidents_with_weather.parquet`

## 6 — Feature Engineering

> 🔄 **TODO** — Geplante Feature-Transformationen:
> - Zyklische Zeitfeatures: `sin/cos(USTUNDE * 2π/24)`, `sin/cos(UMONAT * 2π/12)`
> - `ULAND` aus `UKREIS[:2]` ableiten
> - Interaktion: `is_weekend` × `is_night` (H3)
> - Wetter-Features aus DWD-Join: Temperatur, Sichtweite, Phänomen-Code
> - One-Hot-Encoding für UART, UTYP1 (oder ordinale Behandlung)
> - **Output:** bereinigte Feature-Matrix X + Label-Vektor y für A³-Phase